# Exploratívna Analýza Dát (EDA)

**Kaggle House Prices Dataset**

Tento notebook obsahuje exploratívnu analýzu Kaggle House Prices dataset.

## Ciele:
1. Pochopenie dát a ich distribúcie
2. Identifikácia missing values
3. Analýza target premennej (SalePrice)
4. Korelačná analýza features
5. Vizualizácie pre insights

In [ ]:
# Imports
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"✅ Imports successful")
print(f"   pandas: {pd.__version__}")
print(f"   numpy: {np.__version__}")

## 1. Načítanie Dát

Dáta stiahneme z Kaggle alebo načítame lokálne.

In [ ]:
# Data paths
DATA_DIR = "../data"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

# Check if data exists
if not os.path.exists(TRAIN_PATH):
    print("❌ Data not found!")
    print("\nDownload data using:")
    print("  python scripts/download_kaggle_data.py")
    print("\nOr manually from:")
    print("  https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data")
else:
    print(f"✅ Data found at {TRAIN_PATH}")

In [ ]:
# Load data
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nColumns: {train.shape[1]}")
print(f"Train samples: {train.shape[0]}")
print(f"Test samples: {test.shape[0]}")

## 2. Základný Prehľad Dát

In [ ]:
# First few rows
train.head()

In [ ]:
# Data types
print("Data types:")
print(train.dtypes.value_counts())
print(f"\nNumeric features: {train.select_dtypes(include=[np.number]).shape[1]}")
print(f"Categorical features: {train.select_dtypes(include=['object']).shape[1]}")

In [ ]:
# Summary statistics
train.describe()

## 3. Missing Values Analýza

In [ ]:
# Missing values count
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(f"Features with missing values: {len(missing)}")
print(f"\nTop 10 features with most missing values:")
print(missing.head(10))

In [ ]:
# Visualize missing values
if len(missing) > 0:
    plt.figure(figsize=(12, 6))
    missing_pct = (missing / len(train)) * 100
    missing_pct.plot(kind='bar')
    plt.title('Missing Values Percentage by Feature')
    plt.xlabel('Feature')
    plt.ylabel('Missing %')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("✅ No missing values in dataset")

## 4. Target Premenná: SalePrice

In [ ]:
# SalePrice statistics
print("SalePrice Statistics:")
print(train['SalePrice'].describe())
print(f"\nSkewness: {train['SalePrice'].skew():.2f}")
print(f"Kurtosis: {train['SalePrice'].kurtosis():.2f}")

In [ ]:
# SalePrice distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original distribution
axes[0].hist(train['SalePrice'], bins=50, edgecolor='black')
axes[0].set_title('SalePrice Distribution (Original)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(train['SalePrice'].mean(), color='red', linestyle='--', label='Mean')
axes[0].axvline(train['SalePrice'].median(), color='green', linestyle='--', label='Median')
axes[0].legend()

# Log-transformed distribution
axes[1].hist(np.log1p(train['SalePrice']), bins=50, edgecolor='black')
axes[1].set_title('SalePrice Distribution (Log-transformed)')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Q-Q plot
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original
stats.probplot(train['SalePrice'], dist="norm", plot=axes[0])
axes[0].set_title('Q-Q Plot (Original)')

# Log-transformed
stats.probplot(np.log1p(train['SalePrice']), dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot (Log-transformed)')

plt.tight_layout()
plt.show()

print("💡 Log transformation makes SalePrice more normally distributed")

## 5. Numerické Features Analýza

In [ ]:
# Numeric features (vylúčime Id a target)
numeric_features = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_features.remove('Id')
numeric_features.remove('SalePrice')

print(f"Numeric features: {len(numeric_features)}")
print(numeric_features[:10])  # First 10

In [ ]:
# Correlation with SalePrice (top 15)
correlations = train[numeric_features + ['SalePrice']].corr()['SalePrice'].sort_values(ascending=False)
top_correlations = correlations[1:16]  # Exclude SalePrice itself

plt.figure(figsize=(10, 6))
top_correlations.plot(kind='barh')
plt.title('Top 15 Features Correlated with SalePrice')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

print("\nTop 10 correlations:")
print(top_correlations.head(10))

In [ ]:
# Scatter plots for top correlated features
top_features = correlations[1:5].index.tolist()  # Top 4

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, feature in enumerate(top_features):
    axes[i].scatter(train[feature], train['SalePrice'], alpha=0.5)
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('SalePrice')
    axes[i].set_title(f'{feature} vs SalePrice (r={correlations[feature]:.3f})')
    
    # Add regression line
    z = np.polyfit(train[feature].fillna(0), train['SalePrice'], 1)
    p = np.poly1d(z)
    axes[i].plot(train[feature], p(train[feature]), "r--", alpha=0.8)

plt.tight_layout()
plt.show()

## 6. Korelačná Matica

In [ ]:
# Correlation matrix (top correlated features)
top_10_features = correlations[1:11].index.tolist()
corr_matrix = train[top_10_features + ['SalePrice']].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Top 10 Features + SalePrice')
plt.tight_layout()
plt.show()

## 7. Key Insights

### Zistenia z EDA:

1. **Target Variable (SalePrice)**:
   - Pravostranná (right-skewed) distribúcia
   - Log-transformácia robí distribúciu normálnejšou
   - Vhodné použiť `log1p(SalePrice)` ako target

2. **Missing Values**:
   - Niektoré features majú vysoký % missing values
   - Potrebná imputation stratégia

3. **Najdôležitejšie Features** (top correlations):
   - OverallQual
   - GrLivArea  
   - GarageCars
   - GarageArea
   - TotalBsmtSF

4. **Multikolinearita**:
   - GarageCars a GarageArea sú vysoko korelované (r ≈ 0.88)
   - TotalBsmtSF a 1stFlrSF sú vysoko korelované
   - Možno použiť len jeden z páru features

## 8. Ďalšie Kroky

1. **Feature Engineering**: Vytvoriť nové odvdené features
2. **Missing Values**: Implementovať imputation stratégiu
3. **Categorical Features**: One-hot encoding alebo target encoding
4. **Outliers**: Identifikovať a spracovať outliers
5. **Model Training**: Trénovať model s MLflow trackingom

Pokračuj v `02_feature_engineering.ipynb`